In [1]:
# 1 — сжатие(2) 
import os, json, zlib, bz2, lzma, base64
import pandas as pd
try:
    import brotli
except ImportError:
    brotli = None

# === опции ===
CONTENT_TYPE = "notebook"   # "notebook" или "dataframe"
NB_PATH      = "bootstrap_to_excel.ipynb"   # <-- укажите путь к ноутбуку
SOURCE_DF    = None          # если CONTENT_TYPE == "dataframe" — присвойте нужный df
# =============

_ALGO_NAMES = {"Z": "zlib", "B": "bz2", "L": "lzma", "R": "brotli"}

def _compress_best(raw):
    candidates = {"Z": zlib.compress(raw, level=9),
                  "B": bz2.compress(raw, compresslevel=9),
                  "L": lzma.compress(raw, preset=9 | lzma.PRESET_EXTREME)}
    if brotli is not None:
        candidates["R"] = brotli.compress(raw, quality=11)
    tag = min(candidates, key=lambda k: len(candidates[k]))
    return tag, candidates[tag]

if CONTENT_TYPE == "notebook":
    if not NB_PATH:
        raise ValueError("Укажите путь к ноутбуку в переменной NB_PATH")
    if not os.path.isfile(NB_PATH):
        raise FileNotFoundError(f"Файл не найден: {NB_PATH!r}")
    print(f"Ноутбук: {NB_PATH}")
    with open(NB_PATH, "r", encoding="utf-8") as f:
        nb = json.load(f)
    parts = []
    for i, cell in enumerate(nb["cells"], start=1):
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        parts.append(f"### CELL {i} [{cell['cell_type']}] ###\n{source}\n")
    raw = "".join(parts).encode("utf-8")
    content_tag = "N"
    print(f"Ячеек: {len(nb['cells'])}")

elif CONTENT_TYPE == "dataframe":
    if SOURCE_DF is None:
        raise ValueError("CONTENT_TYPE == 'dataframe', но SOURCE_DF не задан")
    raw = SOURCE_DF.to_csv(index=False).encode("utf-8")
    content_tag = "D"
    print(f"Датафрейм: {SOURCE_DF.shape[0]} строк, {SOURCE_DF.shape[1]} колонок")

else:
    raise ValueError(f"неизвестный CONTENT_TYPE: {CONTENT_TYPE!r}")

algo_tag, compressed = _compress_best(raw)
_qr_prefix  = content_tag + algo_tag
_qr_payload = base64.b64encode(compressed).decode("ascii")
print(f"Сжатие: {len(raw)} -> {len(compressed)} байт ({_ALGO_NAMES[algo_tag]}), base64: {len(_qr_payload)} симв.")

Ноутбук: bootstrap_to_excel.ipynb
Ячеек: 1
Сжатие: 2550 -> 1191 байт (zlib), base64: 1588 симв.


In [2]:
# 2 — партиции или QR-коды

import io
import qrcode
from IPython.display import display, Image

# === опция ===
OUTPUT_MODE = "text"   # "qr" или "text"
# =============

if OUTPUT_MODE == "qr":
    max_chunk = 2953 - len(f"{_qr_prefix}|001/001|")
elif OUTPUT_MODE == "text":
    max_chunk = 2950
else:
    raise ValueError(f"неизвестный OUTPUT_MODE: {OUTPUT_MODE!r}")

chunks = [_qr_payload[i:i + max_chunk] for i in range(0, len(_qr_payload), max_chunk)]
total = len(chunks)
print(f"Частей: {total}")

for idx, chunk in enumerate(chunks, start=1):
    part = f"{_qr_prefix}|{idx:03d}/{total:03d}|{chunk}"
    print(f"--- часть {idx}/{total} ---")
    if OUTPUT_MODE == "qr":
        qr = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_L, box_size=8, border=4)
        qr.add_data(part.encode("ascii"))
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        display(Image(data=buf.getvalue()))
    else:
        print(part)

Частей: 1
--- часть 1/1 ---
NZ|001/001|eNqVVs1uGzcQvu9TEBEK7iLM1lJkOza6h9i1TjkEaIseBGGx2uVKRLjLBZey/AMDjgO0hwbIoQXaS4H2DYSgRow09jPsvlGH3B/91EVcWYaGw5mPM9+QQ3Y6HXR49OIF6qJhKCI6Qp1Ox2JJJqRCIqNpdnrCrViKpB25uTrlNEe10UCkiqCXgVJUpgPGOUHPOZukCdX6AyEjKgn6hkV0A2amGG9RJlT5oeCzJPU51VCWNR8jb2n9vZCvxkK8sh1rnsPEfOwGoWLHFIauYopTUOLnB0/Kq+K6fF1eYWtgNJIFHFvWlAYQiPYc4uLP8rJYFO+L2+Ia/m+Ka0xwioqPxV1xW16Vl64Zt0jEeLgbBpVu1egPAP2gAcsfigUmFlp+8HcZZ7FCX2i7X4rfUM/dbuW93WqQPTkO+Ixq9a8Asyh/hNg+FXcav3yLR1YKFEEGnKZ2nY9j6fwTKifUDynnuZ2rQCpfirnXJagaVMTqMU2jZkqL9YSG1bS6GsHukq7J5br4UL4p/i7fQooLCOAntMJusUD2S8HyXKQIyqJyJYPM2UcwdwmefwFN78p3evgGMvgE5po24OcWGFoUH4Ghm+IGO24M2wdS0rvIToOEegMCgDzyvpUzCgmwM+p1+5CmmrIUDPVGss0O9LBWYYIgeiE9fLCr/7Bjjc2eA9tq89mcxsrTtgRJNpk2shJZLY2FUiIxA1goFhKFBE0RrEfTGXAbKNrwTVDX2TeF1VzpfViz9pQYJ6ed+1xmddQD88GrfswAr5woG+eCswhSjSeHlVtvsN3f21l1C5pDB77tAbSnQrIziCDgHg5BQSWgzKFWvqInyoSygtEyVwmWlbOJ/6B4DneOBodHEE8qHuwDDv2vdQ6GcgbVEXNNuqT5jKvcj2KXgTtoc7sm/cQPATaK/WAycbkIh5U4xOo0o/5EilmGR54HLkMcQtpScDwiCI9Z6